# Inverting ESIS Level-1 Data with MART

This report is a first end-to-end sketch of inverting ESIS flight data with the
multiplicative algebraic reconstruction technique (MART),
using the real, fitted instrument model rather than an idealized cartoon.

The pipeline is:

1. Load a single frame of the Level-1 flight data,
   :func:`esis.flights.f1.data.level_1`.
2. Load the best-fit distortion model of the instrument,
   :func:`esis.flights.f1.optics.distortion_fit`.
3. Linearize the fitted optical system with
   :meth:`optika.systems.SequentialSystem.linearize`,
   which fits polynomial distortion, vignetting, and effective-area models
   to the raytrace and assembles a fast, regridding-based forward model,
   :class:`optika.systems.LinearSystem`.
4. Wrap the linear system in a :class:`ctis.instruments.OptikaInstrument`,
   which adapts it to the CTIS instrument interface used by the inverters.
5. Invert the frame with :class:`ctis.inverters.MartInverter`.

The reconstructed scene targets the three brightest lines in the ESIS passband:
He I 584 Å, Mg X 610 Å, and O V 630 Å,
each with a ±200 km/s velocity window.

**Environment note:** this notebook currently requires unreleased branches of
two sibling libraries, installed editable:
``optika`` on ``feature/interpolated-system`` (`optika PR #152
<https://github.com/sun-data/optika/pull/152>`_)
and ``ctis`` on ``feature/electron-based-instruments`` (`ctis PR #18
<https://github.com/sun-data/ctis/pull/18>`_).

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.constants
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis
import esis

## Loading the Level-1 Observation

Load the Level-1 images from the 2019 flight and select the ``time=15`` frame,
the same frame the distortion fit was optimized against.

In [ ]:
%%time
l1 = esis.flights.f1.data.level_1()
frame = l1[dict(time=15)]
frame.outputs.shape

Display all four channels of the selected frame.

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="channel",
        nrows=frame.shape["channel"],
        figsize=(8, 16),
        sharex=True,
        constrained_layout=True,
        origin="upper",
    )
    na.plt.pcolormesh(
        frame.inputs.pixel.x,
        frame.inputs.pixel.y,
        C=frame.outputs.value,
        ax=axs,
        vmax=np.percentile(frame.outputs.value, 99),
    )
    na.plt.text(
        x=0.5,
        y=1.01,
        s=frame.channel,
        transform=na.plt.transAxes(axs),
        ax=axs,
        ha="center",
        va="bottom",
    )
    na.plt.set_aspect("equal", ax=axs)

## The Fitted Instrument Model

:func:`esis.flights.f1.optics.distortion_fit` returns the ESIS-I design with
the best-fit per-channel distortion parameters applied,
so the raytrace of this instrument reproduces the geometry of the flight data.

In [ ]:
instrument = esis.flights.f1.optics.distortion_fit(num_distribution=0)
system = instrument.system

## The Scene Grid

The reconstructed scene targets the three brightest lines in the ESIS passband,
He I 584 Å, Mg X 610 Å, and O V 630 Å.
The Mg X 609.79 Å line is one component of the Li-like Mg X resonance doublet;
the other component, Mg X 624.94 Å, is fainter and is left out of this first
reconstruction.

In [ ]:
spectrum = esis.flights.f1.spectrum
wavelength_center = na.ScalarArray(
    u.Quantity(
        [
            spectrum.He_I.wavelength,
            spectrum.Mg_X.wavelength,
            spectrum.O_V.wavelength,
        ]
    ),
    axes="line",
)

Give each line a ±200 km/s velocity window and concatenate the three windows
into a single logical ``wavelength`` axis.
A single wavelength axis lets MART invert all three lines jointly,
which is exactly the ESIS disentangling problem:
each detector pixel mixes contributions from every line.

Note that concatenating the windows creates two spurious "gap" cells
between consecutive line windows.
They are retained for now (their reconstructed radiance should stay small)
and will be excluded more carefully in a future iteration.

In [ ]:
num_velocity = 10

velocity = na.linspace(
    start=-200 * u.km / u.s,
    stop=200 * u.km / u.s,
    axis="wavelength",
    num=num_velocity + 1,
)

wavelength_vertices = wavelength_center * (1 + velocity / astropy.constants.c)
wavelength_vertices = wavelength_vertices.to(u.AA).combine_axes(
    axes=("line", "wavelength"),
    axis_new="wavelength",
)
wavelength_vertices.shape

The spatial grid spans the field of view of the instrument.
The pitch is kept deliberately coarse for this first sketch
(a few arcseconds per scene cell, versus the 0.77 arcsec/pix plate scale)
to keep the weight matrix small.

In [ ]:
num_field = 64

field = system.rayfunction_default.inputs.field

position = na.Cartesian2dVectorLinearSpace(
    start=field.min(),
    stop=field.max(),
    axis=na.Cartesian2dVectorArray("field_x", "field_y"),
    num=num_field + 1,
)

coordinates_scene = na.SpectralPositionalVectorArray(
    wavelength=wavelength_vertices,
    position=position,
)

## Linearizing the Optical System

:meth:`~optika.systems.SequentialSystem.linearize` raytraces the fitted system
on the wavelength grid defined above and fits polynomial distortion,
vignetting, and effective-area models to the result.
The returned :class:`~optika.systems.LinearSystem` images scenes by
conservative regridding instead of raytracing,
which is what makes iterative inversion affordable.

In [ ]:
%%time
linear = system.linearize(
    wavelength=wavelength_vertices,
    degree=2,
)

As a sanity check, distort the rest wavelength of each spectral line across the
field of view and overlay the resulting footprints on the Level-1 frame.
Each line's footprint should land on its corresponding image of the field stop.

In [ ]:
probe = na.SpectralPositionalVectorArray(
    wavelength=wavelength_center,
    position=na.Cartesian2dVectorLinearSpace(
        start=field.min(),
        stop=field.max(),
        axis=na.Cartesian2dVectorArray("probe_x", "probe_y"),
        num=11,
    ),
)
footprint = linear.distortion.distort(probe).position

position_sensor = linear.coordinates_sensor

with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="channel",
        nrows=frame.shape["channel"],
        figsize=(8, 16),
        sharex=True,
        constrained_layout=True,
        origin="upper",
    )
    na.plt.pcolormesh(
        position_sensor.x,
        position_sensor.y,
        C=frame.outputs.value,
        ax=axs,
        cmap="gray",
        vmax=np.percentile(frame.outputs.value, 99),
    )
    colors = ["red", "orange", "yellow"]
    labels = ["He I 584", "Mg X 610", "O V 630"]
    for i in range(len(labels)):
        na.plt.scatter(
            footprint.x[dict(line=i)],
            footprint.y[dict(line=i)],
            ax=axs,
            color=colors[i],
            s=4,
            label=labels[i],
        )
    na.plt.set_aspect("equal", ax=axs)
    axs.ndarray[0].legend(loc="upper right", fontsize=8)

## A CTIS Instrument from the Linear System

:class:`ctis.instruments.OptikaInstrument` adapts the linear system to
the CTIS instrument interface used by the inverters:
it computes the conservative-regridding weights that map scene cells to
detector pixels (and their transpose) and exposes the
:meth:`~ctis.instruments.AbstractInstrument.image` /
:meth:`~ctis.instruments.AbstractInstrument.backproject` pair that MART
iterates with.

In [ ]:
mart_instrument = ctis.instruments.OptikaInstrument(
    system=linear,
    coordinates_scene=coordinates_scene,
    channel=frame.channel,
    axis_channel="channel",
    axis_wavelength="wavelength",
    axis_scene_xy=("field_x", "field_y"),
)

Building the weights is the most expensive step of the setup.

In [ ]:
%%time
weights, shape_weights_input, shape_weights_output = mart_instrument.weights
shape_weights_input, shape_weights_output

## Preparing the Images

The observed images are the Level-1 electrons evaluated on the sensor
coordinates of the linear system.
Negative pixels (noise fluctuations about zero after background subtraction)
are clipped, since MART reconstructs a non-negative scene.

In [ ]:
images = na.FunctionArray(
    inputs=mart_instrument.coordinates_sensor,
    outputs=np.maximum(frame.outputs, 0),
)

As a first look at the inversion geometry, backproject the observed images
into the scene grid.
Backprojection is not an inverse—it smears each pixel's electrons across
every scene cell that could have contributed—but the smallest backprojection
across the four channels is already a crude reconstruction.

In [ ]:
%%time
backprojected = mart_instrument.backproject(images.outputs)

In [ ]:
smin = np.min(backprojected.outputs, axis="channel")

i_OV = dict(wavelength=slice(2 * (num_velocity + 1), 3 * (num_velocity + 1) - 1))
map_OV = smin[i_OV].sum("wavelength")

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.pcolormesh(
        coordinates_scene.position.x,
        coordinates_scene.position.y,
        C=map_OV.value,
        ax=ax,
        vmin=0,
        vmax=np.percentile(map_OV.value, 99.5),
    )
    ax.set_aspect("equal")
    ax.set_title("min-over-channels backprojection, O V 630 window")

## Running MART

Invert with an all-ones initial guess,
scaled so the total forward-modeled signal matches the total observed signal.

In [ ]:
unit_scene = backprojected.outputs.unit

shape_scene = {
    ax: num
    for ax, num in shape_weights_input.items()
    if ax != mart_instrument.axis_channel
}

ones = na.ScalarArray(
    ndarray=np.ones(tuple(shape_scene.values())),
    axes=tuple(shape_scene),
)

In [ ]:
%%time
image_ones = mart_instrument.image(ones * unit_scene, noise=False)
scale = images.outputs.sum() / image_ones.outputs.sum()
guess = ones * scale * unit_scene

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=mart_instrument,
    intermediate=True,
    num_iteration=10,
)

In [ ]:
%%time
inversion = mart(images, guess=guess, verbose=True)

## Results

Plot the mean chi-squared of each channel as a function of iteration.

In [ ]:
axis_iteration = mart.axis_iteration

iteration = na.arange(
    start=0,
    stop=inversion.mean_chi_squared.shape[axis_iteration],
    axis=axis_iteration,
)

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(
        iteration,
        inversion.mean_chi_squared,
        ax=ax,
        axis=axis_iteration,
    )
    ax.set_xlabel("iteration")
    ax.set_ylabel(r"mean $\chi^2$")
    ax.set_yscale("log")

Display the reconstructed intensity of each spectral line.

In [ ]:
solution = inversion.solutions[{axis_iteration: ~0}]

labels = ["He I 584", "Mg X 610", "O V 630"]

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        nrows=1,
        figsize=(14, 5),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for i, ax in enumerate(axs.flat):
        j = dict(
            wavelength=slice(i * (num_velocity + 1), (i + 1) * (num_velocity + 1) - 1)
        )
        map_i = solution.outputs[j].sum("wavelength")
        na.plt.pcolormesh(
            coordinates_scene.position.x,
            coordinates_scene.position.y,
            C=map_i.value,
            ax=ax,
            vmin=0,
            vmax=np.percentile(map_i.value, 99.5),
        )
        ax.set_aspect("equal")
        ax.set_title(labels[i])

The reconstruction clearly recovers solar network structure,
but this first pass also shows the artifacts expected from such a coarse
setup: the flux of each line is only partially disentangled from the others
(complementary bands of emission split between the line maps),
the cells at the edge of the field absorb unmodeled flux,
and the mean :math:`\chi^2` is still far from unity when the iteration stops.
More iterations, a finer scene grid, and a vignetting/effective-area model
validated against the flight radiometry should all improve this
(see the next steps below).

Plot the field-averaged spectrum of the reconstruction.

In [ ]:
spectrum_mean = solution.outputs.mean(("field_x", "field_y"))

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        coordinates_scene.wavelength,
        spectrum_mean,
        ax=ax,
        axis="wavelength",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"mean radiance ({ax.get_ylabel()})")

## Residuals

Project the reconstructed scene back through the forward model and subtract it
from the observed images.
Structure remaining in the residuals is signal the reconstruction failed to
capture (or signal the forward model cannot represent),
so this is the most direct diagnostic of both the inversion and the
instrument model.

In [ ]:
%%time
predicted = mart_instrument.image(solution.outputs, noise=False, uncertainty=True)
residual = images.outputs - predicted.outputs.nominal

Display the forward-modeled images of each channel with the same color scale
as the Level-1 frame shown at the top of this report,
so the model and the data can be compared directly before looking at their
difference.

In [ ]:
vmax = np.percentile(frame.outputs.value, 99)

with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="channel",
        nrows=frame.shape["channel"],
        figsize=(8, 16),
        sharex=True,
        constrained_layout=True,
        origin="upper",
    )
    na.plt.pcolormesh(
        position_sensor.x,
        position_sensor.y,
        C=predicted.outputs.nominal.value,
        ax=axs,
        cmap="gray",
        vmin=0,
        vmax=vmax,
    )
    na.plt.text(
        x=0.5,
        y=1.01,
        s=frame.channel,
        transform=na.plt.transAxes(axs),
        ax=axs,
        ha="center",
        va="bottom",
        color="black",
    )
    na.plt.set_aspect("equal", ax=axs)

Display the residuals of each channel,
saturated at the 99th percentile of the observed images so they can be
compared with the Level-1 frame shown at the top of this report.

In [ ]:
vmax = np.percentile(frame.outputs.value, 99)

with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="channel",
        nrows=frame.shape["channel"],
        figsize=(8, 16),
        sharex=True,
        constrained_layout=True,
        origin="upper",
    )
    na.plt.pcolormesh(
        position_sensor.x,
        position_sensor.y,
        C=residual.value,
        ax=axs,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    na.plt.text(
        x=0.5,
        y=1.01,
        s=frame.channel,
        transform=na.plt.transAxes(axs),
        ax=axs,
        ha="center",
        va="bottom",
    )
    na.plt.set_aspect("equal", ax=axs)

Normalize the residuals by the standard deviation of the modeled measurement
noise and plot their distribution for each channel.
A perfect reconstruction of a perfectly-modeled instrument would give a
standard normal distribution;
excess width or asymmetry measures the structure left in the residual maps
above.
The noise model includes shot, Fano, and partial-charge-collection noise per
wavelength, plus the readout noise of the cameras added once per readout.

In [ ]:
residual_standard = residual / predicted.outputs.width

histogram = na.histogram(
    residual_standard,
    bins=dict(bin=101),
    min=-10,
    max=10,
    axis=("detector_x", "detector_y"),
)

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        histogram.inputs,
        histogram.outputs,
        ax=ax,
        axis="bin",
    )
    ax.set_yscale("log")
    ax.set_xlabel("standardized residual")
    ax.set_ylabel("number of pixels")

## Next Steps

- **Time dependence** — repeat the inversion for every Level-1 frame using
  :func:`esis.flights.f1.optics.distortion_fit` with ``axis_time`` so the
  per-frame payload pointing is included in the forward model.
- **Resolution** — the scene grid here is deliberately coarse; push the
  spatial pitch toward the 0.77 arcsec/pix plate scale and refine the velocity
  sampling.
- **Gap cells** — exclude the spurious wavelength cells between line windows
  instead of retaining them in the reconstruction.
- **Validation** — forward-model a synthetic scene
  (:func:`esis.flights.f1.data.synth.scene_iris` or
  :func:`~esis.flights.f1.data.synth.scene_aia`) through the same linear
  system and invert it, so the reconstruction can be compared against a known
  truth.